# Ablation Ladder + Bootstrap Significance

**The defence table.** Two jobs:

1. **Isolate what each component contributes** — hierarchy, weighting, ensemble — by walking a ladder
   where exactly one thing changes at a time.
2. **Say which differences are real.** The project has been quoting four decimals
   (0.5807 → 0.6302 → 0.6339) with Worms at 48 test rows and Analysis at 77. The claim that
   "differences below ~±0.005 are not meaningful" has been an assertion throughout. This measures it.

Right now **v3 (0.6339) over v2 (0.6302) cannot be defended** if a panel asks. After this notebook,
every comparison either has a confidence interval behind it or is explicitly reported as within noise.

## The ladder

| # | Configuration | Isolates |
|---|---|---|
| 1 | Flat XGBoost, `balanced` | the original project baseline |
| 2 | Flat XGBoost, α=0.25 | **weighting**, on the flat problem |
| 3 | Two-stage, `balanced` | **hierarchy**, at fixed weighting |
| 4 | Two-stage, α=0.25 | weighting *inside* the hierarchy |
| 5 | Two-stage + α-diverse ensemble | **the ensemble** |

1→2 and 3→4 measure weighting; 1→3 and 2→4 measure the hierarchy; 4→5 measures the ensemble.

## Protocol

**All five configurations train on exactly the same 195,950 rows.** The ensemble additionally uses
DSEL to fit 63 combiner parameters — negligible against the shared training budget, and it is the
only way to fit a combiner honestly. Published single-model numbers used `train+val` and so run
~0.003 higher; that offset applies uniformly and does not affect the comparisons here.

**Bootstrap:** 2,000 **stratified** resamples of the test set — resampling within each class keeps
$n_c$ fixed, matching the stratified split design. All configurations are scored on the *same*
resample each iteration, so differences are **paired** and their intervals are much tighter than
differences of independent intervals would suggest.

> Runtime ~8 min (the two flat 10-class models dominate: 400 rounds × 10 classes each).

## 1. Setup

In [1]:
%pip install xgboost imbalanced-learn -q

import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_tr,  X_val,  y_tr,  y_val  = train_test_split(X_trf, y_trf, test_size=0.2, random_state=RS, stratify=y_trf)
X_dsel, _, y_dsel, _ = train_test_split(X_val, y_val, test_size=0.5, random_state=RS, stratify=y_val)

y_test_arr = y_test.values
print("train %d | DSEL %d | test %d | features %d"
      % (len(y_tr), len(y_dsel), len(y_test), X.shape[1]))

def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

def sample_w(ya, wc):
    w = wc[ya].astype(float)
    return w * len(w) / w.sum()

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

attack_labels = sorted(l for l in y_tr.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC = len(attack_labels)

def attack_only(Xa, ya):
    m = ya != 0
    return Xa[m].reset_index(drop=True), ya[m].map(fam_to_idx).reset_index(drop=True)

X_tr_att,   y_tr_att   = attack_only(X_tr, y_tr)
X_dsel_att, y_dsel_att = attack_only(X_dsel, y_dsel)
n_att = np.bincount(y_tr_att.values, minlength=NC)
n_10  = np.bincount(y_tr.values, minlength=10)

Note: you may need to restart the kernel to use updated packages.
train 195950 | DSEL 24494 | test 61235 | features 44


## 2. Train the ladder

Each configuration produces one length-61,235 prediction vector on the test set. Everything after this
cell operates on those vectors alone.

In [2]:
PRED = {}

# --- 1. flat XGBoost, balanced ---
t = time.time()
m = make_xgb(); m.fit(X_tr, y_tr, sample_weight=compute_sample_weight("balanced", y_tr))
PRED["1 flat XGB, balanced"] = m.predict(X_test)
print("1 done (%.0fs)" % (time.time() - t))

# --- 2. flat XGBoost, alpha=0.25 ---
t = time.time()
m = make_xgb(); m.fit(X_tr, y_tr, sample_weight=sample_w(y_tr.values, wc_alpha(n_10, 0.25)))
PRED["2 flat XGB, a=0.25"] = m.predict(X_test)
print("2 done (%.0fs)" % (time.time() - t))

# --- shared gate ---
t = time.time()
gate = make_xgb()
gate.fit(X_tr, (y_tr != 0).astype(int),
         sample_weight=compute_sample_weight("balanced", (y_tr != 0).astype(int)))
gate_test = gate.predict(X_test)
print("gate done (%.0fs)" % (time.time() - t))

def route(fam_all):
    yp = np.zeros(len(gate_test), int)
    g = gate_test == 1
    yp[g] = np.vectorize(idx_to_fam.get)(fam_all[g])
    return yp

# --- 3. two-stage, balanced ---
m = make_xgb(); m.fit(X_tr_att, y_tr_att,
                      sample_weight=compute_sample_weight("balanced", y_tr_att))
PRED["3 two-stage, balanced"] = route(m.predict(X_test))

# --- 4. two-stage, alpha=0.25 ---
m = make_xgb(); m.fit(X_tr_att, y_tr_att,
                      sample_weight=sample_w(y_tr_att.values, wc_alpha(n_att, 0.25)))
PRED["4 two-stage, a=0.25"] = route(m.predict(X_test))
print("3 and 4 done")

1 done (42s)
2 done (51s)
gate done (4s)
3 and 4 done


In [3]:
# --- 5. two-stage + alpha-diverse ensemble (7 members, per-class W fitted on DSEL) ---
t = time.time()
POOL = []
for a in (0.0, 0.25, 0.5, 1.0):
    mm = make_xgb(); mm.fit(X_tr_att, y_tr_att, sample_weight=sample_w(y_tr_att.values, wc_alpha(n_att, a)))
    POOL.append(mm)
strat = {c: 2000 for c in range(NC) if n_att[c] < 2000}
Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(X_tr_att, y_tr_att)
mm = make_xgb(); mm.fit(Xs, ys); POOL.append(mm)
for mdl in (RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS),
            ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)):
    mdl.fit(X_tr_att, y_tr_att, sample_weight=sample_w(y_tr_att.values, wc_alpha(n_att, 0.25)))
    POOL.append(mdl)
M = len(POOL)
print("pool of %d fitted (%.0fs)" % (M, time.time() - t))

P_dsel = [p.predict_proba(X_dsel_att) for p in POOL]
P_test = [p.predict_proba(X_test)     for p in POOL]

def fuse(probas, W):
    S = np.zeros(probas[0].shape)
    for i in range(M):
        S += probas[i] * W[i][None, :]
    return S

GRID = (0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0)
yd = y_dsel_att.values
W = np.ones((M, NC)) / M
best = f1_score(yd, fuse(P_dsel, W).argmax(1), average="macro", zero_division=0)
for _ in range(3):
    for c in range(NC):
        for i in range(M):
            cur = W[i, c]
            for g in GRID:
                W[i, c] = g
                if W[:, c].sum() == 0:
                    continue
                s = f1_score(yd, fuse(P_dsel, W).argmax(1), average="macro", zero_division=0)
                if s > best + 1e-7:
                    best, cur = s, g
            W[i, c] = cur
print("W fitted on DSEL, Stage-B macro-F1 %.4f" % best)

PRED["5 two-stage + ensemble"] = route(fuse(P_test, W).argmax(1))
CONFIGS = list(PRED)
print("\npoint estimates:")
for k in CONFIGS:
    print("  %-26s %.4f" % (k, f1_score(y_test_arr, PRED[k], average="macro", zero_division=0)))

pool of 7 fitted (93s)
W fitted on DSEL, Stage-B macro-F1 0.6278

point estimates:
  1 flat XGB, balanced       0.5801
  2 flat XGB, a=0.25         0.5947
  3 two-stage, balanced      0.5740
  4 two-stage, a=0.25        0.6125
  5 two-stage + ensemble     0.6302


## 3. Stratified bootstrap

Resampling **within each class** keeps $n_c$ fixed, which matches how the test set was constructed
(stratified split). All configurations are scored on the same resample each iteration, so every
difference below is **paired** — the correct and much tighter comparison.

In [4]:
def macro_f1_fast(yt, yp, C=10):
    cm = np.bincount(yt * C + yp, minlength=C * C).reshape(C, C)
    tp = np.diag(cm).astype(float)
    den = 2 * tp + (cm.sum(0) - tp) + (cm.sum(1) - tp)
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0)
    return f1.mean(), f1

# sanity: must match sklearn
for k in CONFIGS[:1]:
    assert abs(macro_f1_fast(y_test_arr, PRED[k])[0]
               - f1_score(y_test_arr, PRED[k], average="macro", zero_division=0)) < 1e-9
print("fast macro-F1 matches sklearn")

B = 2000
FINAL = CONFIGS[-1]
rng = np.random.default_rng(RS)
cls_idx = [np.where(y_test_arr == c)[0] for c in range(10)]

boot    = np.zeros((B, len(CONFIGS)))
boot_pc = np.zeros((B, 10))
t = time.time()
for b in range(B):
    idx = np.concatenate([rng.choice(ci, size=len(ci), replace=True) for ci in cls_idx])
    yt = y_test_arr[idx]
    for j, k in enumerate(CONFIGS):
        mf, pc = macro_f1_fast(yt, PRED[k][idx])
        boot[b, j] = mf
        if k == FINAL:
            boot_pc[b] = pc
print("%d resamples in %.0fs" % (B, time.time() - t))

fast macro-F1 matches sklearn
2000 resamples in 4s


## 4. Macro-F1 with 95% confidence intervals

In [5]:
rows = []
for j, k in enumerate(CONFIGS):
    pt = macro_f1_fast(y_test_arr, PRED[k])[0]
    lo, hi = np.percentile(boot[:, j], [2.5, 97.5])
    rows.append(dict(configuration=k, macro_F1=round(pt, 4),
                     ci_low=round(lo, 4), ci_high=round(hi, 4),
                     ci_width=round(hi - lo, 4)))
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
print("\nmean 95%% CI width: %.4f  -> differences smaller than ~%.3f are not resolvable"
      % (tab.ci_width.mean(), tab.ci_width.mean() / 2))

         configuration  macro_F1  ci_low  ci_high  ci_width
  1 flat XGB, balanced    0.5801  0.5644   0.5961    0.0317
    2 flat XGB, a=0.25    0.5947  0.5756   0.6134    0.0378
 3 two-stage, balanced    0.5740  0.5592   0.5894    0.0302
   4 two-stage, a=0.25    0.6125  0.5933   0.6309    0.0376
5 two-stage + ensemble    0.6302  0.6113   0.6490    0.0377

mean 95% CI width: 0.0350  -> differences smaller than ~0.017 are not resolvable


## 5. Paired differences — what each component actually contributes

`p` is the two-sided bootstrap proportion of resamples on the wrong side of zero. A CI excluding zero
means the component's contribution is established; one straddling zero means it is not, however large
the point estimate looks.

In [6]:
COMPARISONS = [
    ("weighting (flat)",        "2 flat XGB, a=0.25",     "1 flat XGB, balanced"),
    ("hierarchy (balanced)",    "3 two-stage, balanced",  "1 flat XGB, balanced"),
    ("hierarchy (a=0.25)",      "4 two-stage, a=0.25",    "2 flat XGB, a=0.25"),
    ("weighting (two-stage)",   "4 two-stage, a=0.25",    "3 two-stage, balanced"),
    ("ensemble",                "5 two-stage + ensemble", "4 two-stage, a=0.25"),
    ("FULL: final vs baseline", "5 two-stage + ensemble", "1 flat XGB, balanced"),
]
rows = []
for tag, a, b in COMPARISONS:
    ja, jb = CONFIGS.index(a), CONFIGS.index(b)
    d = boot[:, ja] - boot[:, jb]
    obs = macro_f1_fast(y_test_arr, PRED[a])[0] - macro_f1_fast(y_test_arr, PRED[b])[0]
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    rows.append(dict(contribution=tag, delta=round(obs, 4),
                     ci_low=round(lo, 4), ci_high=round(hi, 4),
                     p=round(min(p, 1.0), 4),
                     significant="YES" if lo > 0 or hi < 0 else "no"))
print(pd.DataFrame(rows).to_string(index=False))

           contribution   delta  ci_low  ci_high     p significant
       weighting (flat)  0.0146  0.0001   0.0276 0.049         YES
   hierarchy (balanced) -0.0061 -0.0144   0.0030 0.174          no
     hierarchy (a=0.25)  0.0178  0.0061   0.0311 0.005         YES
  weighting (two-stage)  0.0385  0.0246   0.0512 0.000         YES
               ensemble  0.0177  0.0067   0.0301 0.003         YES
FULL: final vs baseline  0.0502  0.0391   0.0606 0.000         YES


### 5.1 Is v3 over v2 defensible?

The published step from 0.6302 to 0.6339 came from cross-fitting (v3), not from anything in this
ladder. Its size can still be judged against the resolution measured here: if +0.0037 sits well inside
the paired-difference intervals above, it cannot be claimed as an improvement.

In [7]:
j = CONFIGS.index(FINAL)
sd = boot[:, j].std()
print("bootstrap SD of the final model's macro-F1 : %.4f" % sd)
print("published v3 - v2 difference               : +0.0037")
print("that is %.2f bootstrap SDs" % (0.0037 / sd))
print("\n-> %s" % ("v3 over v2 is WITHIN NOISE and must not be claimed as an improvement"
                   if 0.0037 < 1.96 * sd else
                   "v3 over v2 exceeds the noise scale"))
print("   (v2 and v3 differ only in cross-fitting, so a paired test would be tighter than this,")
print("    but the marginal SD is the right order-of-magnitude check.)")

bootstrap SD of the final model's macro-F1 : 0.0095
published v3 - v2 difference               : +0.0037
that is 0.39 bootstrap SDs

-> v3 over v2 is WITHIN NOISE and must not be claimed as an improvement
   (v2 and v3 differ only in cross-fitting, so a paired test would be tighter than this,
    but the marginal SD is the right order-of-magnitude check.)


## 6. Per-class intervals for the final model

The direct evidence for the "do not over-interpret rare-class numbers" caveat that the project has
been asserting.

In [8]:
pt_pc = macro_f1_fast(y_test_arr, PRED[FINAL])[1]
rows = []
for c in range(10):
    lo, hi = np.percentile(boot_pc[:, c], [2.5, 97.5])
    rows.append(dict(cls=NAMES[c], support=int((y_test_arr == c).sum()),
                     F1=round(pt_pc[c], 3), ci_low=round(lo, 3), ci_high=round(hi, 3),
                     ci_width=round(hi - lo, 3)))
pc = pd.DataFrame(rows)
print(pc.to_string(index=False))
print("\nwidest intervals (least trustworthy per-class numbers):")
print(pc.sort_values("ci_width", ascending=False).head(4)[["cls", "support", "F1", "ci_width"]].to_string(index=False))

           cls  support    F1  ci_low  ci_high  ci_width
        Benign    45001 0.983   0.982    0.984     0.002
      Analysis       77 0.427   0.361    0.493     0.132
      Backdoor       90 0.652   0.556    0.740     0.184
           DoS      883 0.433   0.401    0.462     0.061
      Exploits     6182 0.750   0.740    0.758     0.018
       Fuzzers     5279 0.708   0.702    0.715     0.013
       Generic      899 0.790   0.771    0.809     0.038
Reconnaissance     2406 0.641   0.627    0.656     0.029
     Shellcode      370 0.360   0.317    0.405     0.088
         Worms       48 0.557   0.422    0.683     0.260

widest intervals (least trustworthy per-class numbers):
      cls  support    F1  ci_width
    Worms       48 0.557     0.260
 Backdoor       90 0.652     0.184
 Analysis       77 0.427     0.132
Shellcode      370 0.360     0.088


## 7. Findings

**The full stack is solidly established (+0.0502, p < 0.001). Two of its three components are not what
the project has been claiming.**

| Contribution | Δ macro-F1 | 95% CI | p | established |
|---|---|---|---|---|
| **weighting (inside two-stage)** | **+0.0385** | [0.0246, 0.0512] | <0.001 | **YES** |
| hierarchy (at α=0.25) | +0.0178 | [0.0061, 0.0311] | 0.005 | **YES** |
| ensemble | +0.0177 | [0.0067, 0.0301] | 0.003 | **YES** |
| weighting (flat) | +0.0146 | [0.0001, 0.0276] | 0.049 | marginal |
| **hierarchy (at `balanced`)** | **−0.0061** | [−0.0144, 0.0030] | 0.174 | **no** |
| **FULL: final vs baseline** | **+0.0502** | [0.0391, 0.0606] | <0.001 | **YES** |

### 1. The hierarchy only works *with* tempered weighting — an interaction, not a main effect

At `balanced` weighting the two-stage hierarchy is **negative and not significant** (−0.0061,
p = 0.174). At α=0.25 it is **+0.0178, p = 0.005**. The same architectural change helps or hurts
depending entirely on the weighting it sits on.

**This corrects an early project claim.** The two-stage work reported flat XGBoost 0.5807 → two-stage
+ balanced 0.5858 and treated +0.005 as evidence the hierarchy helped. That difference is inside the
noise band measured here, and on this run the sign is actually reversed (0.5801 → 0.5740). *The
hierarchy on its own was never established.*

The thesis contribution should therefore be framed as **"hierarchical detection with tempered
class weighting"** — one combined mechanism — not as a hierarchy that works and a weighting scheme
bolted on. That is a more accurate claim and, being an interaction effect, a more interesting one.

### 2. Weighting is the largest single lever, ahead of the ensemble

+0.0385 inside the two-stage, against +0.0177 for the entire seven-member ensemble. The
cheapest change in the project — one exponent on the class weights — contributes more than twice what
the most elaborate one does.

Consistent with the whole record: every gain has come from weighting or from data, never from model
capacity.

### 3. Resolution — two different yardsticks, do not mix them

- **Reporting a single configuration:** mean 95% CI width **0.0350**, i.e. ±0.017. Macro-F1 should be
  quoted as e.g. *0.630 [0.611, 0.649]*, not to four decimals.
- **Comparing two configurations on the same test set:** the **paired** interval is what applies, and
  it is roughly **±0.012** — tighter, because the two configurations see identical resamples. This is
  why +0.0177 can be significant while a marginal CI of ±0.017 might suggest otherwise.

Using marginal widths to judge differences would be the natural mistake here, and it would wrongly
discard two established results.

### 4. v3 over v2 is within noise — keep v3, but not on score grounds

+0.0037 is **0.39 bootstrap SDs** (SD = 0.0095). It cannot be claimed as an improvement.

v3 should still be the reported final model, justified on **protocol** grounds: cross-fitting uses all
of `train+val` for the base models and fits the combiner on ~32,000 rows rather than 6,494, which is
the methodologically sound way to spend the data. \"It scored 0.0037 higher\" is not a defensible
justification and should not appear in the thesis.

By the same measure, the headline progression **0.6157 → 0.6302 → 0.6339** contains only *one*
resolvable step (the ensemble, +0.0145 ≈ the paired resolution) and one that is not.

### 5. Per-class intervals — the rare-class caveat, now measured

| Class | support | F1 | 95% CI | width |
|---|---|---|---|---|
| Benign | 45,001 | 0.983 | [0.982, 0.984] | 0.002 |
| Fuzzers | 5,279 | 0.708 | [0.702, 0.715] | 0.013 |
| Exploits | 6,182 | 0.750 | [0.740, 0.758] | 0.018 |
| Reconnaissance | 2,406 | 0.641 | [0.627, 0.656] | 0.029 |
| Generic | 899 | 0.790 | [0.771, 0.809] | 0.038 |
| DoS | 883 | 0.433 | [0.401, 0.462] | 0.061 |
| Shellcode | 370 | 0.360 | [0.317, 0.405] | 0.088 |
| Analysis | 77 | 0.427 | [0.361, 0.493] | 0.132 |
| Backdoor | 90 | 0.652 | [0.556, 0.740] | 0.184 |
| **Worms** | **48** | **0.557** | **[0.422, 0.683]** | **0.260** |

Worms spans **±0.13** — the point estimate carries roughly one significant figure. Every per-class
comparison the project has made on Worms (v2 0.557 vs v3 0.486, and the "Worms +0.142" gain
attributed to the v2 ensemble) sits comfortably inside this interval and none of them is established.

**Report rare-class results as intervals.** It is both more honest and more defensible than three
decimals, and it pre-empts the obvious question about 48 test rows.

### 6. Consistency check

Configuration 5 reproduces the v2 ensemble exactly (0.6302, and per-class Worms 0.557 / Backdoor 0.652
/ Analysis 0.427 matching the v2 report), confirming the ladder is measuring the intended system.

### What to change in the write-up

1. Frame the contribution as **hierarchy + tempered weighting jointly**; drop any standalone
   "two-stage improves detection" claim.
2. Report macro-F1 with confidence intervals; drop the fourth decimal.
3. Justify v3 as final on **protocol**, not score.
4. Report rare-class F1 as intervals, and stop making per-class comparisons that the intervals cannot
   support.
5. Keep the headline: **+0.0502 [0.039, 0.061] over the baseline, p < 0.001** — that one is real.

### Scope

These intervals capture **test-set sampling only**. Variance from the train/test split and from model
seeds is excluded and would widen them further, so every interval here is optimistic — the safe
direction, but worth stating.
